# 01 · Dataset Exploration & Preparation
**Owner: Teddy**

Part of the EcoSort Waste Management System group project.

## Environment setup (works in **VS Code** and **Google Colab**)
Run this cell first. It detects where you are, gets the code + dataset, and installs deps.
- **VS Code / local:** it uses the repo files already on disk. Put `realwaste.zip` in `data/` (or let it auto-download).
- **Colab:** it clones the repo and downloads the dataset straight to Colab's fast local disk (no Google Drive needed).

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
DATASET_URL = ""  # <-- paste the GitHub *Release* asset URL for realwaste.zip here

if IN_COLAB:
    # 1. get the code
    if not os.path.exists("Ecosort-Waste-Management-System"):
        !git clone https://github.com/REPLACE_ME/Ecosort-Waste-Management-System.git
    %cd Ecosort-Waste-Management-System
    # 2. install deps
    !pip -q install -r requirements.txt
    # 3. get the dataset onto Colab local disk (fast) — not Drive
    if not os.path.exists("data/RealWaste"):
        if DATASET_URL:
            !wget -q -O data/realwaste.zip "$DATASET_URL"
        else:
            print("Set DATASET_URL (GitHub Release asset) or upload data/realwaste.zip")

# make `src` importable from the repo root in both environments
ROOT = os.getcwd()
if os.path.basename(ROOT) == "notebooks":
    ROOT = os.path.dirname(ROOT); os.chdir(ROOT)
sys.path.insert(0, ROOT)

from src import config as C, data_prep
print("Ready. Categories:", C.CATEGORIES)


## Part 1 — EDA & data pipelines
Explore images, text, and policies; build the splits every other part uses.

In [ ]:
data_prep.ensure_images_extracted()
counts = data_prep.image_class_counts()
counts

In [ ]:
import matplotlib.pyplot as plt, pandas as pd
pd.Series(counts).sort_values().plot.barh(title='Images per category'); plt.tight_layout(); plt.show()
print('Class imbalance ratio:', max(counts.values())/min(counts.values()))

In [ ]:
df = data_prep.load_descriptions(split=False)
print(len(df),'descriptions'); df['category'].value_counts()

In [ ]:
# sample images per class
import matplotlib.pyplot as plt, os
from PIL import Image
fig,ax=plt.subplots(3,3,figsize=(9,9))
for a,cat in zip(ax.ravel(),C.CATEGORIES):
    d=os.path.join(C.IMAGE_DIR,cat); f=os.listdir(d)[0]
    a.imshow(Image.open(os.path.join(d,f))); a.set_title(cat); a.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
policies = data_prep.load_policies()
print(len(policies),'policy docs'); print(policies[0]['policy_type'])

### TODO (Teddy)
- Comment on resolution/background/quality challenges
- Note class imbalance & how each model will address it
- Document data splits & any biases/limitations